# Problem 2: Similarity Search using Sentence Embeddings

We build a simple semantic similarity search system using:

- AG News Dataset (first 2000 samples)
- Pretrained embedding model: `sentence-transformers/all-MiniLM-L6-v2`

We will:
1. Compute sentence embeddings
2. Compute cosine similarity
3. Retrieve top-k most similar news texts
4. Build a reusable search function

In [12]:
!pip install datasets sentence-transformers

In [13]:
import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 1️⃣ Loading the AG News Dataset

We use the HuggingFace `ag_news` dataset.

We select the first 2000 samples from the training set.
Each sample contains:
- Title
- Description

In [14]:
dataset = load_dataset("ag_news")
train_data = dataset["train"]

# Take first 2000 samples
subset = train_data.select(range(2000))

# Combine title + description
texts = [item["text"] for item in subset]

print("Number of texts:", len(texts))

Number of texts: 2000


## 2️⃣ Loading Pretrained Sentence Embedding Model

We use:

`sentence-transformers/all-MiniLM-L6-v2`

This model:
- Produces 384-dimensional embeddings
- Is lightweight and fast
- Is widely used for semantic search

In [15]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
embeddings = model.encode(texts, convert_to_numpy=True)

print("Embedding shape:", embeddings.shape)

Embedding shape: (2000, 384)


In [17]:
print(embeddings[:5])

[[ 0.00743864  0.02856238  0.04109551 ... -0.13397597 -0.09019157
   0.0646931 ]
 [-0.00054744 -0.1444023  -0.05263957 ... -0.09893096  0.01725998
   0.06443575]
 [-0.00559561 -0.02277449  0.07513674 ... -0.07831725  0.00378904
   0.06476948]
 [ 0.0079434  -0.08021668  0.07207923 ...  0.04463274  0.0331801
   0.02578293]
 [-0.06110715 -0.04299478  0.07979133 ... -0.07337304  0.01544182
   0.06616858]]


## 4️⃣ Cosine Similarity

To measure semantic similarity between two embeddings:

$$
\text{cosine similarity} =
\frac{\mathbf{e}_1 \cdot \mathbf{e}_2}
{\|\mathbf{e}_1\| \|\mathbf{e}_2\|}
$$

Values range from:
- 1 → very similar
- 0 → unrelated
- -1 → opposite

In [18]:
similarity_matrix = cosine_similarity(embeddings)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (2000, 2000)


## 5️⃣ Query-Based Retrieval

Example query:

query = "new technology for artificial intelligence"

Steps:
1. Compute query embedding
2. Compute similarity with all dataset embeddings
3. Sort and retrieve top 5 results

In [19]:
query = "new technology for artificial intelligence"

query_embedding = model.encode([query], convert_to_numpy=True)[0]

# Compute similarity with all dataset embeddings
similarities = cosine_similarity([query_embedding], embeddings)[0]

# Get top 5 indices
top_indices = np.argsort(similarities)[-5:][::-1]

print("Query:", query)
print("\nTop 5 Results:\n")

for idx in top_indices:
    print(f"Score: {similarities[idx]:.4f}")
    print(texts[idx])
    print("-" * 50)

Query: new technology for artificial intelligence

Top 5 Results:

Score: 0.4742
NASA Develops Robust Artificial Intelligence for Planetary Rovers NASA is planning to add a strong dose of artificial intelligence (AI) to planetary rovers to make them much more self-reliant, capable of making basic decisions during a mission. Scientists are developing very complex AI software that enables a higher level of robotic intelligence.
--------------------------------------------------
Score: 0.3965
Computers with multiple personalities The jury's still out on whether a computer can ever truly be intelligent, but there's no question that it can have multiple personalities. It's just a matter of software.
--------------------------------------------------
Score: 0.3843
New NASA Supercomputer to Aid Theorists and Shuttle Engineers (SPACE.com) SPACE.com - NASA researchers have teamed up with a pair of Silicon Valley firms to build \  a supercomputer that ranks alongside the world's largest Linux-ba

## 6️⃣ Reusable Search Function

We now create a general search function:

def search(query, texts, embeddings, model, top_k=5):



This function:

Encodes query

Computes cosine similarity

Returns top-k results

In [20]:
# 🟦 Code: Search Function
def search(query, texts, embeddings, model, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)[0]

    similarities = cosine_similarity([query_embedding], embeddings)[0]

    top_indices = np.argsort(similarities)[-top_k:][::-1]

    results = []

    for idx in top_indices:
        results.append((texts[idx], similarities[idx]))

    return results

In [21]:
queries = [
    "stock market performance",
    "sports championship results",
    "new medical treatment"
]

for q in queries:
    print("\n==============================")
    print("Query:", q)
    print("==============================\n")

    results = search(q, texts, embeddings, model, top_k=5)

    for text, score in results:
        print(f"Score: {score:.4f}")
        print(text)
        print("-" * 50)


Query: stock market performance

Score: 0.5507
Stocks Higher on Economic Data, Earnings  NEW YORK (Reuters) - U.S. stocks traded higher on Tuesday,  despite oil prices hitting a new high before falling back, as  U.S. economic reports showed an easing of inflationary pressure  and a sharp rebound in the housing market.
--------------------------------------------------
Score: 0.5476
Stocks Higher Despite Soaring Oil Prices NEW YORK - Wall Street shifted higher Monday as bargain hunters shrugged off skyrocketing oil prices and bought shares following an upbeat sales report from Wal-Mart Stores and a bright outlook from Lowe's.    The Dow Jones industrial average was up 84.07, or 0.9 percent, at 9,909.42, after edging 0.1 percent higher last week...
--------------------------------------------------
Score: 0.5244
Stocks Up on Earnings, Oil, Economic Data  NEW YORK (Reuters) - U.S. stocks rose on Tuesday, getting a  boost from some strong retail earnings, lower oil prices and  two separat

## 7️⃣ Analysis of Results

### 1️⃣ Relevance

- Query: "stock market performance"
  → Retrieved news articles related to finance, stocks, and markets.

- Query: "sports championship results"
  → Retrieved sports-related headlines.

- Query: "new medical treatment"
  → Retrieved healthcare and medical research news.

This shows embeddings capture semantic meaning rather than just keywords.

---

### 2️⃣ Why Does This Work?

The model was trained so that semantically similar sentences have embeddings close in vector space.

Thus:

$$
\text{similar meaning} \Rightarrow \text{high cosine similarity}
$$

---

### 3️⃣ Limitations Observed

- If wording is very different but meaning is subtle, retrieval may not be perfect.
- Dataset size is small (2000 samples).
- No indexing structure (like FAISS) — brute force search.

---

### Conclusion

We successfully built a basic semantic search engine using:

- Sentence embeddings
- Cosine similarity
- Ranking

This is the core idea behind:
- Modern search engines
- Retrieval-Augmented Generation (RAG)
- Vector databases